In [ ]:
#| default_exp envs.factorized_POMDPs

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

from nbdev import show_doc


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Factorized POMDPs

In [ ]:
#| export
from projective_simulation.envs.core import Abstract_Env
import numpy as np
from numpy.typing import NDArray

In [ ]:
#| export
class Factorized_POMDP(Abstract_Env):
    '''
    A general construtor for factorized POMDPs. \n
    These differ from standard POMDPs in that the transition function indicates the probability distribution over states in each percept category, rather than over all possible percepts 
    '''
    def __init__(self,
                percept_category_sizes: list[int],                      #list of sizes of each percept category
                factorized_observation_function: NDArray[np.float_],    #An NxM array, where N is the number of env states and M = sum(percept_category_sizes)
                transition_function: NDArray[np.float_],                #An NxNxA array, where A is the number of actions. Rows in each slice contain probability distributions.
                initial_state: int = 0                                  #Start state of POMDP
                ):
        '''
        Checks validity of observation and transition function, assigns variables to self.
        '''
        assert len(percept_category_sizes) >= 1, "There must be at least one percept category"
        assert isinstance(factorized_observation_function, np.ndarray), "factorized_observation_function must be a numpy array"
        assert isinstance(transition_function, np.ndarray), "transition_function must be a numpy array"
        assert np.shape(factorized_observation_function)[0] == np.shape(transition_function)[0], "Number of rows in factorized_observation_function must match number of states in transition_function"
        assert np.shape(transition_function)[0] == np.shape(transition_function)[1], "transition_function must be square in its first two dimensions"
        assert transition_function.ndim == 3, "transition_function must be a 3D array"
        assert np.isclose(np.sum(transition_function,axis =1), 1, atol=1e-9).all(), "All rows in transition_function must sum to 1"

        self.percept_category_sizes = percept_category_sizes
        self.category_indexer = np.concatenate([
            np.full(size, idx) for idx, size in enumerate(percept_category_sizes)
        ])

        # Assert that for each state i and each category idx, the sum of probabilities for that category is 1 (or close)
        for i in range(factorized_observation_function.shape[0]):
            for idx in range(len(percept_category_sizes)):
                cat_probs = factorized_observation_function[i, self.category_indexer == idx]
                assert np.isclose(np.sum(cat_probs), 1, atol=1e-9), \
                    f"Probabilities for state {i}, category {idx} do not sum to 1 (sum={np.sum(cat_probs)})"

        self.factorized_observation_function = factorized_observation_function
        self.transition_function = transition_function
        state = initial_state
        super().__init__(state = state)

    def transition(
            self, 
            action: int):
        """
        updates POMDP state using slice of transition_array indicated by 'action'
        """
        if not action in range(np.shape(self.transition_function)[2]):
            raise ValueError("The action input for POMDP transition must be integer valued and within the scope of the transition function")
        transition_probs = self.transition_function[self.state,:, action]
        self.state = np.random.choice(len(transition_probs), p = transition_probs)
    
    def get_observation(self):
        """
        Determine and return a percept for an agent or agents as a function of self.state
        """
        percept = np.full(len(self.percept_category_sizes), -1) #initialize percept with invalid values
        percept_probs = self.factorized_observation_function[self.state,:]
        for k in range(len(self.percept_category_sizes)):
            category_mask = (self.category_indexer == k)
            category_probs = percept_probs[category_mask]
            percept[k] = np.random.choice(len(category_probs), p = category_probs)
     
        return percept
    
    def get_stationary_state_distribution(self):
        """
        Calculate the stationary distribution of the environment's transition function.
        """
             
        # Average over actions to get a single transition matrix
        avg_transition_matrix = np.mean(self.transition_function, axis=2)
        
        # Solve for the stationary distribution: π * P = π
        # This can be done by finding the left eigenvector corresponding to eigenvalue 1
        eigvals, eigvecs = np.linalg.eig(avg_transition_matrix.T)
        
        # Find the index of the eigenvalue that is approximately 1
        stat_dist_index = np.isclose(eigvals, 1)
        
        if not np.any(stat_dist_index):
            raise ValueError("No eigenvalue close to 1 found; check the transition matrix.")
        
        # Get the corresponding eigenvector (stationary distribution)
        stat_dist = np.real(eigvecs[:, stat_dist_index][:,0])
        
        # Normalize to ensure it sums to 1
        stat_dist /= np.sum(stat_dist)
        
        return stat_dist
    

    def stationary_sensor_entropies(self) -> dict[str,float]:
        """
        Return a dict {f"sensor_{k}": H_k} where H_k is the stationary entropy
        of sensor k, i.e. H(X^{(k)}_t) under the stationary distribution.
        """
        pi = self.get_stationary_state_distribution()                         # (S,)
        O  = self.factorized_observation_function                             # (S, V)
        cat = self.category_indexer                                           # (V,)

        # Stationary mixture of emissions over states: p(value_j) = sum_s pi[s] O[s,j]
        mixed = pi @ O                                                        # (V,)

        ent = {}
        for sensor in np.unique(cat):
            cols = (cat == sensor)
            p = mixed[cols]                                                   # (V_sensor,)
            # H = -sum p log2 p (ignore zeros)
            mask = p > 0
            H = -np.sum(p[mask] * np.log2(p[mask]))
            ent[f"sensor_{sensor}"] = H
        return ent


    def predictive_sensor_entropies(self, suppress_warning:bool = False) -> dict[str,float]:
        """
        Return a dict {f"sensor_{k}": H_k} where H_k is the stationary, one-step
        predictive entropy of sensor k, i.e.
            H_k = E_{S_t ~ pi}[ H( X^{(k)}_{t+1} | S_t ) ].
        """
        pi = self.get_stationary_state_distribution()                         # (S,)
        T  = np.mean(self.transition_function , axis = 2)                     # (S, S), row-stochastic
        if not suppress_warning:
            print("WARNING: predictive_sensor_entropies averages over actions for transition probabilites.")
        O  = self.factorized_observation_function                             # (S, V)
        cat = self.category_indexer                                           # (V,)

        # Per-current-state one-step-ahead predictive over values: Q[i, j] = sum_{s'} T[i, s'] O[s', j]
        Q = T @ O                                                             # (S, V)

        ent: dict[str,float] = {}
        for sensor in np.unique(cat):
            cols = (cat == sensor)
            P = Q[:, cols]                                                    # (S, V_sensor)

            # Per-state entropy: H_i = -sum_j P[i,j] log2 P[i,j]
            # (guard against log(0) by zero-masking)
            H_i = -np.sum(np.where(P > 0, P * np.log2(P), 0.0), axis=1)       # (S,)

            # Average over states under stationarity
            H = pi @ H_i
            ent[f"sensor_{sensor}"] = H
        return ent
    

### Example Factorized POMDP

This example factorized POMDP is a two state environment. Each state produces a two-category percept. In state 0, both categories are more likely to be 0. In state 1, both categories are more likely to be 1. The second category has a higher error rate in both states.

In [ ]:
#Set up factorized POMDP
percept_category_sizes = [2, 2]  # Two categories, each with 2 states

factorized_observation_function = np.array(
    [[0.9, 0.1, 0.8, 0.2],  # State 0
     [0.1, 0.9, 0.2, 0.8]]  # State 1
    )  
transition_function = np.array(
    [[0.,1.],
     [1.,0.]])[:,:,np.newaxis] #2-state cycle, single action

example_factorized_env = Factorized_POMDP(percept_category_sizes, factorized_observation_function, transition_function)

#Set up simulation
T = 10
observed_percepts = ["None"] * T #data structure for storing observations

#simulate for T steps and store observations   
for t in range(T):
    observed_percepts[t] = example_factorized_env.get_observation()
    example_factorized_env.transition(0) #use action 0
    
observed_percepts
                                        

[array([0, 0]),
 array([1, 1]),
 array([0, 0]),
 array([1, 1]),
 array([0, 0]),
 array([1, 1]),
 array([0, 0]),
 array([1, 1]),
 array([0, 1]),
 array([1, 1])]

## Prebuilt Factorized POMDPs
### Factorized Noisy Cycle

In [ ]:
#| export
class Factorized_Noisy_Cycle(Factorized_POMDP):
    '''
    A factorized POMPDP with a determinstic cycle between states with uniform error rates for all percept categories
    '''
    def __init__(self,
                 cycle_length: int,               #number of states in the cycle
                 percept_category_count: int,     #number of percept categories
                 category_size: int,              #number of states in each percept category
                 error_rate: float = 0.1,         #error rate for each category
                 initial_state: int = 0
                ):
        '''
        Assigns "correct" percept category states to each environment state, then constructs corresponding transition and observation arrays
        '''

        assert cycle_length >= 2, "cycle_length must be at least 2"
        assert percept_category_count >= 1, "percept_category_count must be at least 1"
        assert category_size >= 2, "category_size must be at least 2"
        assert 0 <= error_rate < 1, "error_rate must be in the interval [0,1)"

        percept_category_sizes = [category_size] * percept_category_count
    
        # Assign "correct" values to states for each percept category, balancing as evenly as possible
        category_assignments:list[NDArray[np.int_]] = []
        rng = np.random.default_rng()
        for k in range(percept_category_count):
            counts = np.full(category_size, cycle_length // category_size)
            counts[:cycle_length % category_size] += 1
            assignments = np.repeat(np.arange(category_size), counts)
            rng.shuffle(assignments)
            category_assignments.append(assignments)

        N = cycle_length
        M = percept_category_count * category_size
        factorized_observation_function = np.zeros((N, M))

        # Fill in the observation function based on category assignments and error rate
        for k in range(percept_category_count):
            for state in range(N):
                correct = category_assignments[k][state]
                idx = k * category_size
                factorized_observation_function[state, idx:idx + category_size] = error_rate / (category_size - 1)
                factorized_observation_function[state, idx + correct] = 1 - error_rate
        
        #create a cyclic shifted diagonal matrix for transition_function
        transition_function = np.roll(np.eye(N = N), shift = 1, axis = 1) #creates 1-shifted identity matrix
        transition_function = transition_function[:,:,np.newaxis] #adds a dimension to allow for a single action

        super().__init__(percept_category_sizes = percept_category_sizes, 
                         factorized_observation_function = factorized_observation_function, 
                         transition_function = transition_function, 
                         initial_state = initial_state)
    
    def get_stationary_state_distribution(self):
        #Because the transition function is a simple cycle, the stationary distribution is uniform
        return np.full(self.transition_function.shape[0], 1/self.transition_function.shape[0])
    
    


### Example Factorized Noisy Cycle

In [ ]:
#Set up factorized POMDP
cycle_length = 5
percept_category_count = 2
category_size = 2
error_rate = 0.1
example_factorized_noisy_cycle = Factorized_Noisy_Cycle(cycle_length, percept_category_count, category_size, error_rate)

#Set up Simulation
T = 20 #total time steps to simulate
observed_percepts = ["None"] * T #data structure for storing observations

print(f'Observation Function: \n {example_factorized_noisy_cycle.factorized_observation_function}')
print('Simulation Results:')
#simulate for T steps and store observations
for t in range(T):
    observed_percepts[t] = example_factorized_noisy_cycle.get_observation()
    print(f'state: {example_factorized_noisy_cycle.state}, percept: {observed_percepts[t]}')
    example_factorized_noisy_cycle.transition(0) #use action 0
    #| export
import warnings
import numpy as np
from typing import List, Dict, Any, Optional




Observation Function: 
 [[0.1 0.9 0.9 0.1]
 [0.1 0.9 0.1 0.9]
 [0.9 0.1 0.9 0.1]
 [0.9 0.1 0.1 0.9]
 [0.9 0.1 0.9 0.1]]
Simulation Results:
state: 0, percept: [1 0]
state: 1, percept: [1 1]
state: 2, percept: [0 0]
state: 3, percept: [0 1]
state: 4, percept: [0 0]
state: 0, percept: [1 0]
state: 1, percept: [1 1]
state: 2, percept: [0 0]
state: 3, percept: [0 1]
state: 4, percept: [0 0]
state: 0, percept: [0 0]
state: 1, percept: [1 1]
state: 2, percept: [0 0]
state: 3, percept: [0 1]
state: 4, percept: [0 0]
state: 0, percept: [1 0]
state: 1, percept: [1 0]
state: 2, percept: [0 0]
state: 3, percept: [0 1]
state: 4, percept: [1 0]


In [ ]:
#| export
import warnings
import numpy as np
from typing import List, Dict, Any, Optional
from projective_simulation.methods.samplers import sample_binomial_gap

class List_Sequencer(Factorized_POMDP):
    """
    A factorized POMDP that produces a specific percept sequence that emulates classic series recognition tasks of human and animal memory
    """
    def __init__(self,
                 item_size: int,                     # number of percept categories per item (K)
                 item_category_size: int,            # number of valid values per category (C), values are 1..C
                 series_length: int,                 # items per series
                 reuse_items: bool = True,            # If true, items can be reused in different lists. Items are never reused for the same list
                 stimulus_duration: int = 1,         # time steps for which an item (or probe stimulus) is presented
                 mean_timeout_interval: float = 4,   # μ: desired mean timeout length (Binomial)
                 var_time_out_interval: float = 2,   # σ²: desired variance (should be < μ for Binomial under-dispersion)
                 num_series: int = 3,                # number of series to present
                 seed: Optional[int] = None,          # random seed for reproducibility (use the same seed to reproduce)
                 enable_probe: bool = False          # if True, adds a state to each percept category for a probe cue stimulus, enabling probe recognition
                 ):
        """
        Build a factorized POMDP that emits: \n
          [series_1 items] + Binomial gap + [series_2 items] + ... + Binomial gap + \n
          [series_{num_series} items] + [retention block] + [probe item]

        Value conventions per percept category (size = item_category_size + 2): \n
            0                        -> timeout token \n
            1..item_category_size    -> real item values \n
            item_category_size + 1   -> retention token

        Transitions are deterministic with a single action (num_actions=1). \n
        Observations are deterministic (one-hot per category).
        """
        # ---------- validation ----------
        if item_size < 1:
            raise ValueError("item_size must be >= 1")
        if item_category_size < 1:
            raise ValueError("item_category_size must be >= 1")
        if series_length < 1 or num_series < 1:
            raise ValueError("series_length and num_series must be >= 1")

        # RNG from seed
        self.seed = seed
        self.rng = np.random.default_rng(self.seed)

        self.mean_timeout_interval = float(mean_timeout_interval)
        self.var_time_out_interval = float(var_time_out_interval)

        # Warn if requested variance is at/above Poisson boundary (Binomial cannot match it)
        if self.var_time_out_interval >= self.mean_timeout_interval:
            warnings.warn(
                "var_time_out_interval >= mean_timeout_interval: "
                "Binomial cannot achieve ≥ Poisson dispersion; sampling approximates Poisson behaviour while enforcing mean."
            )

        # ---------- sample item indices without building the full list of possibilities ----------
        # Total number of distinct items = C^K (each item is a length-K vector with values in 1..C)
        total_items = item_category_size ** item_size  # Python int; we avoid materializing the full set of possible items

        needed_for_series = series_length * num_series

        if reuse_items:
            if series_length > total_items:
                raise ValueError(
                    f"Not enough distinct items: need {series_length}, but only {total_items} possible "
                    f"({item_category_size}^{item_size})."
                )
            
            sampled_idx = np.array([]) #to be filled
            for _ in range(num_series):
                #sample items without replacement for each list, then concatenate lists
                sampled_idx = np.concatenate([sampled_idx, self.rng.choice(total_items, series_length, replace = False)])

        else:
            if needed_for_series > total_items:
                raise ValueError(
                    f"Not enough distinct items: need {needed_for_series}, but only {total_items} possible "
                    f"({item_category_size}^{item_size})."
                )

            # Draw distinct integer IDs in [0, total_items) and decode them to items on-the-fly.
            sampled_idx = self.rng.choice(total_items, size=needed_for_series, replace=False)
        
        sampled_items = self._indices_to_items(sampled_idx, item_size, item_category_size)  # (needed_total, K)

        # Reshape into (num_series, series_length, item_size)
        series_items = sampled_items[:needed_for_series, :].reshape(num_series, series_length, item_size)

        # ---------- tokens and percept sequence ----------
        timeout_token = 0
        percept_sequence: List[NDArray[np.int_]] = []

        # Binomial parameters from (μ, σ²). See notes in _sample_binomial_gap.
        mu = self.mean_timeout_interval
        var = self.var_time_out_interval

        # Assemble: series + gaps
        for s in range(num_series):
            # emit items of series s
            for j in range(series_length):
                for _ in range(stimulus_duration):
                    percept_sequence.append(series_items[s, j, :].copy())

            # inter-series gap except after last series
            if s < num_series - 1:
                gap_len = sample_binomial_gap(mu, var, rng = self.rng)
                for _ in range(gap_len):
                    percept_sequence.append(np.full(item_size, timeout_token, dtype=int))

        # ---------- observation function (deterministic, factorized one-hots) ----------
        per_cat_size = item_category_size + 1 #add one state for timeout
        if enable_probe:
            per_cat_size += 1
        percept_category_sizes = [per_cat_size] * item_size  # same per category
        num_states = len(percept_sequence)
        obs_dim = sum(percept_category_sizes)
        factorized_observation_function = np.zeros((num_states, obs_dim), dtype=float)

        # Slice offsets for each category block in the concatenated observation vector
        row_offset = np.cumsum([0] + percept_category_sizes[:-1])

        def _one_hot(length: int, idx: int) -> NDArray[np.float_]:
            v = np.zeros(length, dtype=float)
            v[idx] = 1.0
            return v

        for i, percept in enumerate(percept_sequence):
            row = np.zeros(obs_dim, dtype=float)
            for k in range(item_size):
                v_k = int(percept[k])  # 0..C+1
                start = row_offset[k]
                row[start:start + per_cat_size] = _one_hot(per_cat_size, v_k)
            factorized_observation_function[i, :] = row

        # ---------- deterministic transitions (num_actions = 1) ----------
        # T[i, i+1, 0] = 1 for all but last state; last state self-loops
        num_actions = 1
        transition_function = np.zeros((num_states, num_states, num_actions), dtype=float)
        for i in range(num_states - 1):
            transition_function[i, i + 1, 0] = 1.0
        transition_function[num_states - 1, 0, 0] = 1.0

        # ---------- parent init ----------
        super().__init__(
            percept_category_sizes=percept_category_sizes,
            factorized_observation_function=factorized_observation_function,
            transition_function=transition_function,
            initial_state=0
        )

        # ---------- persistent attributes for inspection ----------
        self.item_size = item_size
        self.item_category_size = item_category_size
        self.series_length = series_length
        self.num_series = num_series
        self.item_indices = sampled_idx
        self.series_items = series_items                                # (num_series, series_length, item_size)
        self.percept_sequence = np.array(percept_sequence, dtype=int)   # (num_states, item_size)
        self.stimulus_duration = stimulus_duration

    def to_record(self, include_arrays: bool = True) -> Dict[str, Any]:
        """
        Return a dict whose 'constructor_kwargs' can be passed directly to Series_Recognition_Probe(**...).
        If include_arrays=True, also include the realized outputs for convenience.
        """
        record: Dict[str, Any] = {
            "constructor_kwargs": {
                "item_size": self.item_size,
                "item_category_size": self.item_category_size,
                "series_length": self.series_length,
                "stimulus_duration": self.stimulus_duration,
                "mean_timeout_interval": self.mean_timeout_interval,
                "var_time_out_interval": self.var_time_out_interval,
                "num_series": self.num_series,
                "seed": self.seed,
            }
        }
        if include_arrays:
            record["sequence"] = self.percept_sequence.copy()
            record["series_items"] = self.series_items.copy()
        return record

    @classmethod
    def from_record(cls, record: Dict[str, Any]) -> "Series_Recognition_Probe":
        """Recreate the instance from a record produced by to_record() or generate_many()."""
        return cls(**record["constructor_kwargs"])

    @classmethod
    def generate_many(cls,
                      n_sims: int,                        # how many independent sequences to generate
                      item_size: int,
                      item_category_size: int,
                      series_length: int,
                      stimulus_duration: int,
                      mean_timeout_interval: float = 4.0,
                      var_time_out_interval: float = 2.0,
                      num_series: int = 3,
                      master_seed: Optional[int] = None,  # seed that deterministically produces per-sim child seeds
                      return_instances: bool = False,     # if False, return records with constructor_kwargs (+ arrays)
                      include_arrays: bool = True         # if returning records, include realized arrays (percepts, used items, probe)
                      ) -> List[Any]:
        """
        Efficiently generate many independent sequences.
        Returns a list of either instances or records. Each record contains 'constructor_kwargs'
        that can be passed directly to Series_Recognition_Probe(**...).
        """
        # Parent RNG that deterministically produces child seeds
        parent = np.random.default_rng(master_seed)
        child_seeds = parent.integers(0, 2**63 - 1, size=n_sims, dtype=np.int64)

        out: List[Any] = []
        for s in map(int, child_seeds):
            if return_instances:
                out.append(
                    cls(item_size=item_size,
                        item_category_size=item_category_size,
                        series_length=series_length,
                        stimulus_duration = stimulus_duration,
                        mean_timeout_interval=mean_timeout_interval,
                        var_time_out_interval=var_time_out_interval,
                        num_series=num_series,
                        seed=s)
                )
            else:
                inst = cls(item_size=item_size,
                           item_category_size=item_category_size,
                           series_length=series_length,
                           stimulus_duration = stimulus_duration,
                           mean_timeout_interval=mean_timeout_interval,
                           var_time_out_interval=var_time_out_interval,
                           num_series=num_series,
                           seed=s)
                out.append(inst.to_record(include_arrays=include_arrays))
        return out

    # ----- index → item decoder (no need to build the full C^K list) -----

    @staticmethod
    def _indices_to_items(idxs: NDArray[np.int_],     # shape (N,), integers in [0, C^K)
                          item_size: int,             # K: number of categories per item
                          item_category_size: int     # C: values per category (1..C)
                          ) -> NDArray[np.int_]:
        """
        Convert integer IDs into concrete items without constructing all C^K combinations.

        Returns
        -------
        Shape (len(idxs), K). Each row is an item with entries in 1..C.
        """
        C = int(item_category_size)
        K = int(item_size)
        idxs = np.asarray(idxs, dtype=np.int64)  # ensure vectorized integer ops

        # Positional block sizes: [C^(K-1), C^(K-2), ..., C^1, C^0], shape (1, K)
        # Each element, j, of this list is the number of items needed to iterate up the jth digit of an item count in base C
        powers = (C ** np.arange(K-1, -1, -1, dtype=np.int64))[None, :] #final slice creates a column vector to match shape of percept array

        # For each ID n and each position j:
        #   how many full blocks of size powers[j] fit into n?  (n // powers[j])
        #   take that count modulo C to stay within {0..C-1}
        digits_0_based = (idxs[:, None] // powers) % C    # shape (N, K), values 0..C-1

        # shift to 1..C for actual category values
        return digits_0_based + 1


### Example List Sequencer

In [ ]:
example_list_sequence = List_Sequencer(
    item_size = 2,
    item_category_size= 2,
    series_length= 4,
    num_series = 2
)

example_list_sequence.percept_sequence

array([[2, 1],
       [2, 2],
       [1, 1],
       [1, 2],
       [0, 0],
       [0, 0],
       [0, 0],
       [0, 0],
       [0, 0],
       [2, 2],
       [1, 2],
       [2, 1],
       [1, 1]])

In [ ]:
#| export
import numpy as np
from typing import List, Dict, Any, Optional

class Series_Recognition_Probe(List_Sequencer):
    """
    A factorized POMDP that produces a specific percept sequence that emulates classic series recognition tasks of human and animal memory
    """
    def __init__(self,
                 item_size: int,                     # number of percept categories per item (K)
                 item_category_size: int,            # number of valid values per category (C), values are 1..C
                 series_length: int,                 # items per series
                 reuse_items: bool = True,
                 stimulus_duration: int = 1,         # time steps for which an item (or probe stimulus) is presented
                 mean_timeout_interval: float = 4,   # μ: desired mean timeout length (Binomial)
                 var_time_out_interval: float = 2,   # σ²: desired variance (should be < μ for Binomial under-dispersion)
                 retention_interval: int = 4,        # number of retention percept steps after final series
                 probe_cue_duration: int = 2,        # number of time steps for unique cue prior to probe
                 num_series: int = 3,                # number of series to present
                 probe_position: int = -1,           # -1 novel; 0 sample from earlier (not last) series; 1..L index into last series
                 seed: Optional[int] = None          # random seed for reproducibility (use the same seed to reproduce)
                 ):
        """
        Build a factorized POMDP that emits: \n
          [series_1 items] + Binomial gap + [series_2 items] + ... + Binomial gap + \n
          [series_{num_series} items] + [retention block] + [probe item]

        Value conventions per percept category (size = item_category_size + 2): \n
            0                        -> timeout token \n
            1..item_category_size    -> real item values \n
            item_category_size + 1   -> retention token

        Transitions are deterministic with a single action (num_actions=1). \n
        Observations are deterministic (one-hot per category).
        """
        # ---------- validation ----------
        super().__init__(
            item_size = item_size,
            item_category_size=item_category_size,
            series_length=series_length,
            reuse_items=reuse_items,
            stimulus_duration=stimulus_duration,
            mean_timeout_interval=mean_timeout_interval,
            var_time_out_interval=var_time_out_interval,
            num_series=num_series,
            seed = seed,
            enable_probe = True
        )
        if retention_interval < 0:
            raise ValueError("retention_interval must be >= 0")

        total_items = self.item_category_size ** self.item_size
        # ---------- tokens and percept sequence ----------
        timeout_token = 0
        probe_signal_token = item_category_size + 1

        # Retention block
        for _ in range(retention_interval):
            self.percept_sequence = np.concatenate((self.percept_sequence, np.full(item_size, timeout_token, dtype=int)[np.newaxis,:]), axis = 0)
        for _ in range(probe_cue_duration):
            self.percept_sequence = np.concatenate((self.percept_sequence,np.full(item_size, probe_signal_token, dtype=int)[np.newaxis,:]), axis = 0)

        # Probe item placement
        if probe_position == -1:
            novel_indices = [n for n in range(total_items) if n not in self.item_indices]
            probe_idx = self.rng.choice(novel_indices, size = 1)
            probe = self._indices_to_items(idxs = probe_idx, item_size = self.item_size, item_category_size=self.item_category_size)
        elif probe_position == 0:
            if num_series == 1:
                raise ValueError("probe_position=0 requires at least 2 series (to exclude the last).")
            earlier = self.series_items[:num_series - 1].reshape(-1, self.item_size)
            pick = self.rng.integers(low=0, high=earlier.shape[0])
            probe = earlier[pick][np.newaxis,:]
        elif 1 <= probe_position <= series_length:
            probe = self.series_items[-1, probe_position - 1][np.newaxis,:]
        else:
            raise ValueError(f"probe_position must be -1, 0, or in [1, {series_length}]")

        for _ in range(stimulus_duration):
            self.percept_sequence = np.concatenate((self.percept_sequence, probe), axis = 0)

        # ---------- add rows to observation function for retention and probe ----------
        per_cat_size = item_category_size + 2
        percept_category_sizes = [per_cat_size] * item_size  # same per category
        num_prev_states = np.shape(self.factorized_observation_function)[0]
        num_new_states = np.shape(self.percept_sequence)[0] - num_prev_states
        obs_dim = sum(percept_category_sizes)
        self.factorized_observation_function = np.concatenate((self.factorized_observation_function,np.zeros((num_new_states, obs_dim), dtype=float)), axis = 0)

        # Slice offsets for each category block in the concatenated observation vector
        row_offset = np.cumsum([0] + percept_category_sizes[:-1])

        def _one_hot(length: int, idx: int) -> NDArray[np.float_]:
            v = np.zeros(length, dtype=float)
            v[idx] = 1.0
            return v

        for i, percept in enumerate(self.percept_sequence[num_prev_states:]):
            row = np.zeros(obs_dim, dtype=float)
            for k in range(item_size):
                v_k = int(percept[k])  # 0..C+1
                start = row_offset[k]
                row[start:start + per_cat_size] = _one_hot(per_cat_size, v_k)
            self.factorized_observation_function[i, :] = row

        # ---------- deterministic transitions (num_actions = 1) ----------
        # T[i, i+1, 0] = 1 for all but last state; last state self-loops
        num_actions = 1
        num_states = num_prev_states + num_new_states
        self.transition_function = np.zeros((num_states, num_states, num_actions), dtype=float)
        for i in range(num_states - 1):
            self.transition_function[i, i + 1, 0] = 1.0
        self.transition_function[num_states - 1, 0, 0] = 1.0 #sequence repeats

        # ---------- persistent attributes for inspection ----------
        self.retention_interval = retention_interval
        self.probe_position = probe_position
        self.probe_item = probe.copy()   

    def to_record(self, include_arrays: bool = True) -> Dict[str, Any]:
        """
        Return a dict whose 'constructor_kwargs' can be passed directly to Series_Recognition_Probe(**...).
        If include_arrays=True, also include the realized outputs for convenience.
        """
        record: Dict[str, Any] = {
            "constructor_kwargs": {
                "item_size": self.item_size,
                "item_category_size": self.item_category_size,
                "series_length": self.series_length,
                "stimulus_duration": self.stimulus_duration,
                "mean_timeout_interval": self.mean_timeout_interval,
                "var_time_out_interval": self.var_time_out_interval,
                "retention_interval": self.retention_interval,
                "num_series": self.num_series,
                "probe_position": self.probe_position,
                "seed": self.seed,
            }
        }
        if include_arrays:
            record["sequence"] = self.percept_sequence.copy()
            record["series_items"] = self.series_items.copy()
            record["probe_item"] = self.probe_item.copy()
        return record

    @classmethod
    def from_record(cls, record: Dict[str, Any]) -> "Series_Recognition_Probe":
        """Recreate the instance from a record produced by to_record() or generate_many()."""
        return cls(**record["constructor_kwargs"])

    @classmethod
    def generate_many(cls,
                      n_sims: int,                        # how many independent sequences to generate
                      item_size: int,
                      item_category_size: int,
                      series_length: int,
                      stimulus_duration: int,
                      mean_timeout_interval: float = 4.0,
                      var_time_out_interval: float = 2.0,
                      retention_interval: int = 4,
                      probe_cue_duration: int = 2,
                      num_series: int = 3,
                      probe_position: int = -1,
                      master_seed: Optional[int] = None,  # seed that deterministically produces per-sim child seeds
                      return_instances: bool = False,     # if False, return records with constructor_kwargs (+ arrays)
                      include_arrays: bool = True         # if returning records, include realized arrays (percepts, used items, probe)
                      ) -> List[Any]:
        """
        Efficiently generate many independent sequences.
        Returns a list of either instances or records. Each record contains 'constructor_kwargs'
        that can be passed directly to Series_Recognition_Probe(**...).
        """
        # Parent RNG that deterministically produces child seeds
        parent = np.random.default_rng(master_seed)
        child_seeds = parent.integers(0, 2**63 - 1, size=n_sims, dtype=np.int64)

        out: List[Any] = []
        for s in map(int, child_seeds):
            if return_instances:
                out.append(
                    cls(item_size=item_size,
                        item_category_size=item_category_size,
                        series_length=series_length,
                        stimulus_duration = stimulus_duration,
                        mean_timeout_interval=mean_timeout_interval,
                        var_time_out_interval=var_time_out_interval,
                        retention_interval=retention_interval,
                        probe_cue_duration = probe_cue_duration,
                        num_series=num_series,
                        probe_position=probe_position,
                        seed=s)
                )
            else:
                inst = cls(item_size=item_size,
                           item_category_size=item_category_size,
                           series_length=series_length,
                           stimulus_duration = stimulus_duration,
                           mean_timeout_interval=mean_timeout_interval,
                           var_time_out_interval=var_time_out_interval,
                           retention_interval=retention_interval,
                           probe_cue_duration = probe_cue_duration,
                           num_series=num_series,
                           probe_position=probe_position,
                           seed=s)
                out.append(inst.to_record(include_arrays=include_arrays))
        return out


#### Example Series Recognition Probe

In [ ]:
test_series_recognition = Series_Recognition_Probe(
    item_size= 4, 
    item_category_size=3, 
    mean_timeout_interval=2,
    var_time_out_interval=0.5,
    series_length=4, 
    retention_interval=2, 
    num_series=2, 
    probe_position= 3)
test_series_recognition.percept_sequence

array([[2, 3, 3, 2],
       [3, 2, 3, 1],
       [3, 3, 2, 3],
       [1, 1, 3, 2],
       [0, 0, 0, 0],
       [2, 1, 2, 3],
       [2, 2, 3, 2],
       [1, 3, 3, 1],
       [2, 3, 2, 1],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [4, 4, 4, 4],
       [4, 4, 4, 4],
       [1, 3, 3, 1]], dtype=int64)